# Klasy używane w algorytmach

In [125]:
class Point:
    
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __repr__(self):
        return f"Point({self.x}, {self.y})"

class Segment:
    
    def __init__(self, point1, point2):
        self.first_point = point1
        self.second_point = point2
        
    def reversed(self):
        return Segment(self.second_point, self.first_point)

# Rysowanie punktów

In [126]:
%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.widgets import Button

In [127]:
class PolygonDrawer:
    def __init__(self):
        self.fig, self.ax = plt.subplots()
        self.ax.set_title("Kliknij, aby rysować punkty")
        self.points = []
        
        # Buttons
        self.clear_button_ax = self.fig.add_axes([0.81, 0, 0.1, 0.075])
        self.clear_button = Button(self.clear_button_ax, 'Wyczyść')
        self.clear_button.on_clicked(self.clear_polygon)
        
        # Click
        self.cid = self.fig.canvas.mpl_connect('button_press_event', self.onclick)
        self.draw_polygon()

    def onclick(self, event):
        if event.inaxes != self.ax:
            return
        self.points.append(Point(event.xdata, event.ydata))
        self.draw_polygon()

    def draw_polygon(self):
        xlim, ylim = self.ax.get_xlim(), self.ax.get_ylim()
        
        self.ax.clear()
        self.ax.set_title("Kliknij, aby rysować punkty")
        
        if self.points:
            self.ax.scatter([p.x for p in self.points], [p.y for p in self.points], color='blue')
        
        self.ax.set_xlim(xlim)
        self.ax.set_ylim(ylim)
        self.fig.canvas.draw()
    
    def clear_polygon(self, event):
        self.points = []
        self.ax.clear()
        self.ax.set_title("Kliknij, aby rysować punkty")
        self.fig.canvas.draw()


# Wizualizacja przebigu algorytmów

In [128]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import math

In [1]:
class ConvexHullVisualizer:
    def __init__(self):
        
        self.frames = []
        self.speed = 500

    def add_frame(self, points, hulls_closed, hulls_not_closed, segments, current_point):
        frame_data = {
            "points": [(p.x, p.y) for p in points],
            "hulls_closed": [[(p.x, p.y) for p in hull] for hull in hulls_closed],
            "hulls_not_closed": [[(p.x, p.y) for p in hull] for hull in hulls_not_closed],
            "segments": [(segment.first_point.x, segment.first_point.y, segment.second_point.x, segment.second_point.y) for segment in segments],
            "current_point": current_point
        }
        self.frames.append(frame_data)

    def get_frames(self):
        return self.frames
    
    def create_animation(self):
        
        def sort_points_by_angle(points):

            start = min(points, key=lambda p: (p[1], p[0]))

            def angle_from_start(p):
                return math.atan2(p[1] - start[1], p[0] - start[0])

            sorted_points = sorted(points, key=angle_from_start)
            return sorted_points

        def update(frame):
            axs.clear()
            points = frame["points"]
            hulls_closed = frame["hulls_closed"]
            hulls_not_closed = frame["hulls_not_closed"]
            segments = frame["segments"]
            current_point = frame["current_point"]

            axs.scatter([p[0] for p in points], [p[1] for p in points], color='black')

            if hulls_closed:
                
                for hull in hulls_closed:
                    ordered_hull_points = sort_points_by_angle(hull)

                    closed_hull_points = ordered_hull_points + [ordered_hull_points[0]]
                    axs.plot([p[0] for p in closed_hull_points], [p[1] for p in closed_hull_points], color='lightblue', lw=2)
                    axs.scatter([p[0] for p in hull], [p[1] for p in hull], color='lightblue', s=100)
            
            if hulls_not_closed:
                colors = ['pink', 'lightblue', 'lightgreen']
                for i, hull in enumerate(hulls_not_closed):
                    
                    axs.scatter([p[0] for p in hull], [p[1] for p in hull], color=colors[i], s=100)
                    axs.plot([p[0] for p in hull], [p[1] for p in hull], color=colors[i], lw=2)
            
            for seg in segments:
                axs.plot([seg[0], seg[2]], [seg[1], seg[3]], color='lightgreen', lw=2)

            if current_point and hulls_not_closed:
                current_point = (current_point.x, current_point.y)
                axs.scatter(current_point[0], current_point[1], color='lightgreen', s=100)
                axs.plot([hulls_not_closed[0][-1][0], current_point[0]], [hulls_not_closed[0][-1][1], current_point[1]], color='lightgreen', lw=2)
            elif current_point:
                current_point = (current_point.x, current_point.y)
                axs.scatter(current_point[0], current_point[1], color='lightgreen', s=100)

            padding = 0.1
            x_coords = [p[0] for p in points]
            y_coords = [p[1] for p in points]
            x_min, x_max = min(x_coords), max(x_coords)
            y_min, y_max = min(y_coords), max(y_coords)

            axs.set_xlim(x_min - padding * (x_max - x_min), x_max + padding * (x_max - x_min))
            axs.set_ylim(y_min - padding * (y_max - y_min), y_max + padding * (y_max - y_min))

            axs.set_title("Wizualizacja otoczki wypukłej")

        fig, axs = plt.subplots()
        anim = FuncAnimation(fig, update, frames=self.frames, repeat=False, interval=self.speed)
        plt.close(fig)

        return HTML(anim.to_jshtml())
